# Speech → ASL gloss demo (local Jupyter)

**What this notebook does.** Demo, TODO §13 Phase 4: the team's Colab main loop (record →
Whisper → gloss → display) running locally, through `sb.synthesize.pipeline`. No Colab
APIs, no Drive, no signasl.org scraping.

- **§1 live:** record from the microphone, then transcribe and gloss.
- **§2 from a file:** any audio file.
- **§3 typed text:** gloss only (no ASR), to try the engines quickly.

The gloss is shown as chips: **green** = GISLR (our recognizer knows the sign), **blue** =
WLASL only, **orange** = no sign (fingerspell). If WLASL videos are under
`data/raw/wlasl/videos/`, the trimmed clips play under the gloss. Rendering is otherwise
deferred (TODO §13).

No artifacts: outputs are shown, not saved. The engine and Whisper model come from
`configs/speech.pipeline.json`; override them in the setup cell.

## Setup

In [ ]:
# ============================================================
# Imports, configs, load Whisper + the gloss engine
# ============================================================
import json
from pathlib import Path

from IPython.display import HTML, display

from sb.core.paths import EXPERIMENTS_DIR
from sb.synthesize import asr, display as D, lexicon
from sb.synthesize.gloss import GlossEngine
from sb.synthesize.pipeline import SpeechToGloss

ENGINE = None       # None -> config "engine"; or rules_v1 / rules_v2 / hybrid_team / hybrid_guarded
ASR_MODEL = None    # None -> config asr.model; e.g. "large-v3-turbo"

CONFIG_DIR = EXPERIMENTS_DIR / "synthesis" / "configs"
SPEECH = json.loads((CONFIG_DIR / "speech.pipeline.json").read_text(encoding="utf-8"))
T2G = json.loads((CONFIG_DIR / "aslg.text2gloss.json").read_text(encoding="utf-8"))
NB_DIR = EXPERIMENTS_DIR / "synthesis"

PIPE = SpeechToGloss.from_configs(SPEECH, T2G, engine=ENGINE, asr_model=ASR_MODEL)
LEX = lexicon.Lexicon()
print(f"ASR: {PIPE.asr.model_name} on {PIPE.asr.device} ({PIPE.asr.compute_type})")
print(f"gloss engine: {PIPE.engine.name} (T5 {'loaded' if PIPE.engine.uses_t5 else 'not available -> rules'})")


def show(res):
    print(f"English : {res.english or '(no speech detected)'}")
    print(f"Gloss   : {res.gloss.gloss}")
    print(f"ASR {res.asr_seconds * 1000:.0f} ms for {res.audio_seconds:.1f} s of audio · gloss {res.gloss_seconds * 1000:.0f} ms")
    for s in res.gloss.per_sentence:
        if s.get("reasons"):
            print(f"   guard rejected T5 '{s.get('t5')}' ({', '.join(s['reasons'])}) -> {s['source']}")
    if res.gloss.gloss:
        display(HTML(D.lexicon_view(LEX, res.gloss.gloss, NB_DIR)))

## 1. Live: record and translate

In [ ]:
# ============================================================
# Record SECONDS from the default mic -> English -> gloss
# ============================================================
SECONDS = SPEECH["recording"]["seconds"]

print(f"recording {SECONDS} s ... speak now")
audio = asr.record(SECONDS)
show(PIPE(audio))

## 2. From an audio file

In [ ]:
# ============================================================
# Audio file -> English -> gloss
# ============================================================
AUDIO_FILE = None  # e.g. Path("data/raw/speech30/01.wav") (relative to the repo root)

if AUDIO_FILE is None:
    print("set AUDIO_FILE")
else:
    from sb.core.paths import ROOT_DIR

    show(PIPE(ROOT_DIR / AUDIO_FILE))

## 3. Typed text (no ASR): compare the engines

In [ ]:
# ============================================================
# English text -> gloss, every engine side by side
# ============================================================
TEXT = "If it rains tomorrow, I will stay home. Can you help me?"

for name in T2G["eval"]["engines"]:
    g = GlossEngine(name, T2G["t5"])(TEXT)
    print(f"{name:15s} {g.gloss}")
display(HTML(D.lexicon_view(LEX, GlossEngine(PIPE.engine.name, T2G["t5"])(TEXT).gloss, NB_DIR)))